# PowerPoint TTS Movie Builder (Colab T4)

This notebook runs Step 2 (audio synthesis) and Step 3 (video build) on Colab GPU.

Required uploads:
- `*.pptx`
- `*.wav` (reference voice audio)
- `*.txt` (reference transcript)
- `slides.zip` containing `page1.png`, `page2.png`, ...

In [ ]:
# Cell 1: GPU check
!nvidia-smi

In [ ]:
# Cell 2: Clone repository
# Replace with your repository URL
REPO_URL = "https://github.com/mkt-kuno/qwen_tts_pptx"
!git clone {REPO_URL} repo
%cd repo

In [ ]:
# Cell 3: Install dependencies (Colab/T4)
!apt-get -y update
!apt-get -y install ffmpeg sox

!python -m pip install -U pip
!python -m pip install --extra-index-url https://download.pytorch.org/whl/cu124 \
  "cffi>=1.17" "torch==2.9.1" "torchaudio==2.9.1" "torchvision==0.24.1" \
  "gradio>=5,<6" "qwen-tts==0.1.1"

In [ ]:
# Cell 4: Runtime import checks
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from qwen_tts import Qwen3TTSModel
print("qwen_tts import ok")

In [ ]:
# Cell 5: Upload files
# Upload: pptx, wav, txt, slides.zip
from google.colab import files
uploaded = files.upload()
print("uploaded:", list(uploaded.keys()))

In [ ]:
# Cell 6: Prepare workspace and unzip slides
from pathlib import Path
import zipfile
import shutil

ROOT = Path("/content/job")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

for p in Path("/content").iterdir():
    if p.is_file() and p.name in uploaded:
        shutil.copy2(p, ROOT / p.name)

pptx = next(ROOT.glob("*.pptx"))
ref_audio = next(ROOT.glob("*.wav"))
ref_text = next(ROOT.glob("*.txt"))
slides_zip = next(ROOT.glob("*.zip"))

slides_dir = ROOT / "work" / "slides"
slides_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(slides_zip) as zf:
    zf.extractall(slides_dir)

pngs = sorted(slides_dir.glob("page*.png"))
print("pptx:", pptx)
print("ref_audio:", ref_audio)
print("ref_text:", ref_text)
print("slides_zip:", slides_zip)
print("slides count:", len(pngs))
assert len(pngs) > 0, "slides.zip must contain pageN.png files"

In [ ]:
# Cell 7: Step 2 - Audio synthesis
import logging
from app.pipeline.steps import step2_synthesize_audio

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

LANGUAGES = ["JA", "EN", "ZH"]
MODEL_SIZE = "1.7B"
DEVICE = "cuda:0"
DTYPE = "float16"
FORCE_REGENERATE = False

result2 = step2_synthesize_audio(
    pptx_path=pptx,
    project_root=ROOT,
    ref_audio=ref_audio,
    ref_text=ref_text,
    languages=LANGUAGES,
    model_size=MODEL_SIZE,
    device=DEVICE,
    dtype=DTYPE,
    force_regenerate=FORCE_REGENERATE,
)

print("step2 done")
print("slide_count:", result2.slide_count)
print("generated_count:", result2.generated_count)
print("cache_hit_count:", result2.cache_hit_count)
print("active_languages:", [x.tag for x in result2.active_languages])

In [ ]:
# Cell 8: Step 3 - Video build
from app.pipeline.steps import step3_build_videos

SLIDE_PADDING_SEC = 1.5
FPS = 5

result3 = step3_build_videos(
    project_root=ROOT,
    languages=["JA", "EN", "ZH"],
    slide_padding_sec=SLIDE_PADDING_SEC,
    fps=FPS,
)

print("step3 done")
print("per-language outputs:")
for p in result3.per_language_outputs:
    print(" -", p)
print("multilingual:", result3.multilingual_output)

In [ ]:
# Cell 9: Zip outputs and download
from pathlib import Path
import zipfile
from google.colab import files

output_dir = ROOT / "output"
bundle = ROOT / "output_bundle.zip"

with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(ROOT))

print("bundle:", bundle)
files.download(str(bundle))